<a href="https://colab.research.google.com/github/2121045-eng/dongsa/blob/main/dongsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install Streamlit (if not already installed)
!pip install streamlit

import streamlit as st
import random
import time

# --- 페이지 설정 ---
st.set_page_config(page_title="AI 동사 퍼즐", page_icon="🎬")

# --- 앱 세션 상태 초기화 ---
if 'quiz_data' not in st.session_state:
    # 예시: AI 이미지 생성 모델이 만든 GIF 느낌의 샘플 이미지들
    # 실제로는 AI로 생성한 치료용 GIF URL을 여기에 넣으면 됩니다.
    st.session_state.quiz_data = [
        {"verb": "먹다", "url": "https://media.giphy.com/media/xT9IgusfDcqpPFzjdS/giphy.gif", "level": "초급"},
        {"verb": "자다", "url": "https://media.giphy.com/media/l2JhnvO90GQqQ3XQ4/giphy.gif", "level": "초급"},
        {"verb": "달리다", "url": "https://media.giphy.com/media/3o7ZetIsjtbkgNEAJa/giphy.gif", "level": "초급"},
        {"verb": "씻다", "url": "https://media.giphy.com/media/3o84U64219uT7O0k7K/giphy.gif", "level": "중급"},
        {"verb": "그리다", "url": "https://media.giphy.com/media/l41Ys1fZjy45mY46Y/giphy.gif", "level": "중급"}
    ]
    random.shuffle(st.session_state.quiz_data) # 문제 순서 랜덤화
    st.session_state.score = 0
    st.session_state.wrong_answers = []
    st.session_state.current_idx = 0
    st.session_state.game_over = False

# --- 사이드바: 점수 및 프로필 ---
with st.sidebar:
    st.header("🏆 오늘의 점수")
    st.subheader(f"{st.session_state.score} / {len(st.session_state.quiz_data)}")
    st.progress(st.session_state.score / len(st.session_state.quiz_data) if st.session_state.quiz_data else 0)

    st.markdown("---")
    st.info("💡 **치료사 팁:** 아이가 동사를 말하면 여기에 입력해 주세요. 맞으면 자동으로 점수가 올라갑니다!")

# --- 메인 화면: 게임 플레이 ---
st.title("🎬 AI 움직이는 그림 동사 맞히기")
st.write("그림 속 사람이 무엇을 하고 있을까요?")

if not st.session_state.game_over:
    # 현재 문제 가져오기
    current_q = st.session_state.quiz_data[st.session_state.current_idx]

    # 1. 문제 표시
    st.subheader(f"문제 {st.session_state.current_idx + 1}")

    # 레이아웃 나누기: 이미지와 입력창
    col1, col2 = st.columns([2, 1])

    with col1:
        # AI 생성 GIF 표시
        st.image(current_q["url"], caption=f"({current_q['level']}) 무엇을 하고 있나요?")

    with col2:
        st.write(" ") # 여백
        st.write(" ")
        # 2. 정답 입력
        user_answer = st.text_input("아이의 답변을 적어주세요.", key=f"q_{st.session_state.current_idx}")
        check_button = st.button("정답 확인 ✅", key=f"btn_{st.session_state.current_idx}")

    # 3. 채점 로직
    if check_button and user_answer:
        # 입력된 답변과 정답 비교 (공백 제거)
        correct_verb = current_q["verb"].strip()
        user_verb = user_answer.strip()

        if user_verb == correct_verb:
            st.success(f"축하해요! 맞았어요! 🎉 ({correct_verb})")
            st.session_state.score += 1
            st.balloons() # 정답 효과
            time.sleep(1) # 잠시 대기
        else:
            st.error(f"아쉽네요! 정답은 **'{correct_verb}'**였어요.")
            st.session_state.wrong_answers.append({
                "verb": correct_verb,
                "user_answer": user_verb,
                "url": current_q["url"]
            })
            time.sleep(1.5) # 정답 확인 시간 제공

        # 다음 문제로 넘어가기
        if st.session_state.current_idx < len(st.session_state.quiz_data) - 1:
            st.session_state.current_idx += 1
            st.rerun() # 화면 새로고침
        else:
            st.session_state.game_over = True
            st.rerun()

elif st.session_state.game_over:
    # 최종 결과 화면
    st.markdown("---")
    st.header("🎊 연습이 모두 끝났어요!")

    final_score = st.session_state.score
    total_questions = len(st.session_state.quiz_data)

    st.subheader(f"최종 점수: **{final_score} / {total_questions}**")

    if final_score == total_questions:
        st.balloons()
        st.success("와! 모든 문제를 완벽하게 맞췄어요! 참 잘했어요! 👍👍👍")
    elif final_score >= total_questions // 2:
        st.info("많이 맞췄네요! 다음에는 더 잘할 수 있을 거예요! 화이팅! 💪")
    else:
        st.warning("조금 더 연습이 필요해요. 함께 다시 해볼까요?")

    # 오답 분석 리포트
    if st.session_state.wrong_answers:
        st.markdown("---")
        st.subheader("⚠️ 다시 연습하면 좋은 동사 리스트 (오답 노트)")

        # 오답 이미지와 정답/입력값 비교 표
        cols = st.columns(len(st.session_state.wrong_answers))
        for i, wrong in enumerate(st.session_state.wrong_answers):
            with cols[i]:
                st.image(wrong["url"], width=100)
                st.write(f"정답: **{wrong['verb']}**")
                st.write(f"(입력: {wrong['user_answer']})")

    # 다시 시작 버튼
    if st.button("처음부터 다시 하기 🔄"):
        # 세션 상태 초기화
        for key in list(st.session_state.keys()):
            del st.session_state[key]
        st.rerun()

# --- 하단 정보 ---
st.markdown("---")
st.caption("이 앱은 AI 기술(동사별 움직이는 이미지 생성)을 활용하여 언어치료 중 구문론 및 어휘력 훈련을 돕는 교육용 도구입니다. 과제 제출용으로 제작되었습니다.")

2026-04-23 13:55:21.907 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.913 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-23 13:55:21.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()